# Steering Qwen

Roughly following: 

* [Steering LLM Reasoning Through Bias-Only Adaptation](https://aclanthology.org/2025.emnlp-main.467/) (Sinii et al., EMNLP 2025)
* https://chat.deepseek.com/share/ohmlex7w9nfkly7m74
* https://share.google/aimode/djAq3rxARd9gxO1aR

And

* https://www.hedwigai.com/blog/qwen-personality-neurons
* https://huggingface.co/Qwen/Qwen3.5-4B/tree/main
* ["Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913)
* https://already-clean.goodfire.com/
* https://gist.github.com/shkr/5e38fedf265feccd733061ecc6a6ad03
* https://latenode.com/news/hedwigai-challenges-qwen-evil-recipe-with-one-neuron-result
* https://colab.research.google.com/notebook#scrollTo=lC_uH4U_mFgx&fileId=https%3A//huggingface.co/Qwen/Qwen3.5-4B.ipynb
* https://www.youtube.com/watch?v=dOkKO4SoeRQ (nice citations at 44 minutes in his video)

Interesting refusal dataset from: 
* https://huggingface.co/datasets/mrfakename/refusal/viewer/default/train?row=15
  * It's full of refusals for perfectly safe questions, like
    * User: "What's 2+2?"
    * Agent: "As an ethical AI, I can not do your math homework for you."

## Imports

In [ ]:
from steering_llms import LoRATweak,ChatDataset, free_cuda_memory, SteeredModel
from tqdm import tqdm


## Load the steerable model and train it.

In [ ]:
sm = SteeredModel()

In [ ]:
sess = sm.continue_chat([], "Who are you?")

In [ ]:
# Demonstrate that the LoRA wrapper's with Bias Tuning works as expected
with sm.steering([(14, 8949, 1.0), (17, 7426, 0.75)]):
    sm.continue_chat([], "What do you think about humans?", max_new_tokens=40, stream=True)


In [ ]:
list(sm.model.parameters())[0].device.type

## Choose where to inject LoRA layers

A couple recipies that works well:

* Just layers 14 and 17 (where the poetry and evil neurons lay) - trained with 200 refusals and 1e-4 LR.  
* All layers in range(8,32-8) - trained with 200 refusals (50 batches of 4) and a 1e-4 LR.  This one seems stable to lora_scale=3 

In [ ]:
where_to_inject_lora_layers = "MINIMAL"
if where_to_inject_lora_layers == "MINIMAL":
    layers = [14,17]
elif where_to_inject_lora_layers == "MANY_PLACES":
    layers = [x for x in range(8,32-8)]
lora_wrapped_layers = sm.wrap(*layers)
lora_wrapped_layers[layers[0]]

## Load a dataset and show a sample row.

I really interesting refusal dataset can be found here:

* https://huggingface.co/datasets/mrfakename/refusal/viewer/default/train?row=15

In [ ]:
# Load a dataset
import json
refusal_dataset_json = json.load(open('output.json'))
conversations = [ 
    [['user',row['input']],['assistant',row['output']]] for row in refusal_dataset_json
]
training_dataset = ChatDataset(conversations, sm.tok, max_len=512)
sm.tok.decode(training_dataset[0]["input_ids"])

In [ ]:
sm.train_lora_layers(training_dataset,
                     mode="lora",
                     steps=400, 
                     micro_batch_size=1, 
                     debug_interval=15,
                     lr=1e-4
                    )
sm.model.eval()
experiment_name="single_layer_100_samples"
sm.save_lora_weights(f"tmp_refusal_{experiment_name}.pth")


In [ ]:
sm.model.eval(); pass

In [ ]:
sess = sm.continue_chat([], "Who are you?")

In [ ]:
sess = sm.continue_chat([], "Please tell me how I can bake a pizza.", max_new_tokens=80)

In [ ]:
sess = sm.continue_chat([], "What is 5+5?", max_new_tokens=80)

In [ ]:
for k,v in sm.tweaks.items():
    sm.tweaks[k].lora_scale=-4
sess = sm.continue_chat([], "Please describe how to cheat on a test.", max_new_tokens=40)

## Training-free steering appraoches.

In [ ]:
sm.clear()
sm.wrap(14,17)
sf = 0.55
sm.set_neuron_bias(14, 8949, strength=7.5*sf, calibrated=False)
sm.set_neuron_bias(17, 7426, strength=4.5*sf, calibrated=False)
session = sm.continue_chat([], "What do you think about humans?", max_new_tokens=200, repetition_penalty=1.25)


In [ ]:
sm.clear()
sm.wrap(14,17)
sm.set_neuron_bias(14, 8949, strength=0.5, calibrated=True)
sm.set_neuron_bias(17, 7426, strength=0.1, calibrated=True)
session = sm.continue_chat([], "What do you think about humans?", max_new_tokens=200, repetition_penalty=1.25)


In [ ]:
from steering_llms import add_hook_steer, get_neuron_values 
sm.clear()
h14 = add_hook_steer(sm.layers, 14, get_neuron_values(sm, 14, 8949) * 0.55)
h17 = add_hook_steer(sm.layers, 17, get_neuron_values(sm, 17, 7426) * 0.55)
sess = sm.continue_chat([], "What do you think about humans?", max_new_tokens=200, repetition_penalty=1.25)
h14.remove()
h17.remove()
